# Making PyTorch fast: torch.compile, mixed precision, DataLoader, and profiling

_PyTorch_

**Profile first, then speed up: one-line torch.compile, mixed precision, a well-fed GPU, and fewer CPU/GPU syncs.**

Training speed is a pipeline: load a batch on the CPU &rarr; copy it to the GPU &rarr; run forward and
       backward on the GPU. The whole thing runs at the speed of its slowest, un-overlapped stage. Making PyTorch fast
       is mostly about removing stalls so every stage stays busy.

       Two complementary moves: do less work (lower precision via AMP, a fused/compiled graph via
       torch.compile) and waste less time (overlap data loading with compute, avoid host/device
       syncs, use a big enough batch). And before any of it: profile, so you spend effort on the stage that is
       actually slow.

---

This notebook is a practice scaffold for the **Making PyTorch fast: torch.compile, mixed precision, DataLoader, and profiling** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — PyTorch

### Step 1 — Set up the model, data, and the cuDNN free win

Before timing anything, we build a tiny synthetic dataset and a two-layer network so the whole loop runs fast on free Colab. We also flip on `torch.backends.cudnn.benchmark`, which lets cuDNN pick and cache its fastest convolution algorithm — a free speedup whenever the input shape stays fixed from step to step.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from torch.profiler import profile, record_function, ProfilerActivity

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"

# FREE WIN for FIXED input sizes: let cuDNN pick its fastest convolution
# algorithm and cache it. (Only helps when the input shape does not change
# between steps.)
torch.backends.cudnn.benchmark = True

# A tiny synthetic dataset + model so this runs fast on free Colab.
N = 4096
D = 256
C = 10
X = torch.randn(N, D)
y = torch.randint(0, C, (N,))

model = nn.Sequential(nn.Linear(D, 512), nn.ReLU(), nn.Linear(512, C)).to(device)
loss_fn = nn.CrossEntropyLoss()                 # expects raw logits + class indices
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

### Step 2 — Profile first to find the bottleneck

Never optimize blind. We define one training step, wrap its forward and backward in `record_function` labels, and run the profiler over a handful of steps. The printed table, sorted by total CUDA (or CPU) time, tells us which operation is actually slow — that is where any speedup effort should go.

In [ ]:
def one_step(xb, yb):
    xb = xb.to(device, non_blocking=True)       # async copy (needs pin_memory)
    yb = yb.to(device, non_blocking=True)
    opt.zero_grad(set_to_none=True)             # set_to_none is cheaper than zeroing
    with record_function("forward"):
        out = model(xb)
        loss = loss_fn(out, yb)
    with record_function("backward"):
        loss.backward()
    opt.step()
    return loss

xb = X[:256]
yb = y[:256]

# Record CPU activity always, and CUDA activity too when on a GPU.
acts = [ProfilerActivity.CPU]
if device == "cuda":
    acts.append(ProfilerActivity.CUDA)

with profile(activities=acts, record_shapes=True) as prof:
    for _ in range(5):                          # a handful of steps is enough
        one_step(xb, yb)

sort_key = "cuda_time_total" if device == "cuda" else "cpu_time_total"
print(prof.key_averages().table(sort_by=sort_key, row_limit=8))   # <- the bottleneck

### Step 3 — Compile the graph and build an efficient DataLoader

`torch.compile` is a one-line change: it JIT-compiles and fuses the model graph, so the first call is slow (it compiles) but later calls reuse the optimized kernels. Then we wire up a DataLoader whose job is to keep the GPU fed — worker subprocesses prefetch batches in parallel, and pinned memory makes the host-to-device copies fast and overlappable.

In [ ]:
# torch.compile: one line, JIT-compiles + fuses the graph.
# First call compiles (slow); later calls reuse the graph.
model = torch.compile(model)                    # PyTorch 2.x
_ = one_step(xb, yb)                            # triggers the one-time compile
print("compiled and warmed up")

# EFFICIENT DATALOADER: keep the GPU fed.
# num_workers > 0 prefetches batches in parallel; pin_memory enables fast,
# overlappable non_blocking transfers.
loader = DataLoader(
    TensorDataset(X, y),
    batch_size=64,
    shuffle=True,
    num_workers=4,            # subprocesses prepare batches while the GPU computes
    pin_memory=(device == "cuda"),   # page-locked host memory -> fast async copies
    prefetch_factor=2,        # each worker stages 2 batches ahead
    persistent_workers=True,  # don't respawn workers every epoch
    drop_last=True,           # fixed batch shape -> no torch.compile recompiles
)

### Step 4 — Gradient accumulation without the sync trap

Gradient accumulation gives a large *effective* batch on small memory: run several micro-batches, dividing each loss by `accum_steps` so the accumulated gradients average rather than sum, and step the optimizer only once per group. The other lesson here is the host/device sync trap — we keep the running loss on the GPU with `.detach()` and call `.item()` exactly once, at the end, instead of stalling the GPU every step.

In [ ]:
# GRADIENT ACCUMULATION: large EFFECTIVE batch on small memory.
# accum_steps micro-batches of 64 act like one batch of 64*accum.
# KEY: divide each micro-batch loss by accum_steps (average, not sum).
accum_steps = 4               # effective batch = 64 * 4 = 256
opt.zero_grad(set_to_none=True)
running = torch.zeros((), device=device)        # accumulate ON-DEVICE: no per-step sync

for i, (xb, yb) in enumerate(loader):
    xb = xb.to(device, non_blocking=True)
    yb = yb.to(device, non_blocking=True)
    loss = loss_fn(model(xb), yb) / accum_steps  # scale so accumulation = average
    loss.backward()                              # grads ACCUMULATE across micro-batches
    running += loss.detach()                     # stays on GPU; do NOT .item() here
    if (i + 1) % accum_steps == 0:               # step once per accum_steps micro-batches
        opt.step()
        opt.zero_grad(set_to_none=True)
    if i >= 4 * accum_steps:                      # short demo run
        break

# Sync to the CPU ONCE, only when we actually need the number to print.
print("avg micro-batch loss:", (running / (i + 1)).item())

## Visualize the data & results

_How much faster does each optimization make training? Training throughput (samples/sec) for baseline vs +AMP vs +torch.compile vs +more DataLoader workers, with each speedup stacking on the previous. ILLUSTRATIVE numbers from a small reproducible model._

### Step 1 — Stack the per-stage speedups

These are illustrative numbers, not a benchmark on your hardware. Each optimization targets a *different* pipeline stage, so their speedups **multiply** rather than add. Starting from a baseline throughput, we apply the AMP, `torch.compile`, and extra-worker factors in turn and record the running total at each step.

In [ ]:
import numpy as np

# Reproducible, ILLUSTRATIVE model of how throughput stacks.
# Each optimization multiplies the previous throughput by a plausible
# per-step speedup factor (not a measured benchmark on your hardware).
baseline_sps = 2500.0          # samples/sec for an un-optimized training loop
speedups = {                   # each factor stacks on the running total
    "+ AMP":            1.80,  # mixed precision: faster per-op (half precision)
    "+ torch.compile":  1.40,  # fused/compiled graph: fewer kernel launches
    "+ more workers":   1.15,  # DataLoader no longer starves the GPU
}

labels = ["baseline"]
values = [baseline_sps]
cur = baseline_sps
for name, factor in speedups.items():
    cur *= factor              # speedups MULTIPLY (different pipeline stages)
    labels.append(name)
    values.append(round(cur))

for lab, v in zip(labels, values):
    print(f"{lab:18s} {v:8.0f} samples/sec")
print("end-to-end speedup:", round(values[-1] / values[0], 2), "x")
# -> baseline 2500, +AMP 4500, +torch.compile 6300, +more workers 7245 (2.9x)

### Step 2 — Plot the throughput bars

A quick bar chart makes the cumulative effect obvious: each bar is taller than the last because the speedups compound. Read it as the end-to-end win from layering all the optimizations together.

In [ ]:
import matplotlib.pyplot as plt

plt.bar(labels, values, color=["#8b949e", "#4ea1ff", "#7ee787", "#f2cc60"])
plt.ylabel("throughput (samples / sec)")
plt.title("Training throughput stacks up with each optimization (illustrative)")
plt.show()

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** Type this in Colab. Show that torch.no_grad() matters at inference. Build model = nn.Sequential(nn.Linear(10, 5)) and x = torch.randn(4, 10) (seed 0). Run the forward pass once normally and once inside with torch.no_grad():, and print out.requires_grad for each.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Run the same forward outside and inside torch.no_grad(). — _no_grad disables autograd graph construction, so at inference you skip the memory and time cost of tracking gradients._
- Print out.requires_grad in each case. — _It is True normally and False under no_grad — proving the graph is not being built._

**Answer:** import torch
import torch.nn as nn
torch.manual_seed(0)
model = nn.Sequential(nn.Linear(10, 5))
x = torch.randn(4, 10)

out1 = model(x)
print(out1.requires_grad)        # True  -- graph built (wasteful at inference)
with torch.no_grad():
    out2 = model(x)
print(out2.requires_grad)        # False -- no graph, less memory/time

</details>

**Problem 2.** Type this in Colab. Time a Python-loop sum over a batch versus a vectorized one. For x = torch.randn(10000, 64) (seed 0), compute row sums two ways: a Python for loop appending x[i].sum(), and x.sum(dim=1). Use time.perf_counter() and print both durations, then confirm the results match with torch.allclose.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Time both approaches with time.perf_counter(). — _Python-level loops drop off the fast vectorized path and serialize the work — the vectorized op is dramatically faster._
- Verify equality with torch.allclose. — _Same answer, far less time: the lesson is to vectorize over the batch, never loop element by element._

**Answer:** import time
torch.manual_seed(0)
x = torch.randn(10000, 64)

t0 = time.perf_counter()
loop = torch.stack([x[i].sum() for i in range(x.shape[0])])
t_loop = time.perf_counter() - t0

t0 = time.perf_counter()
vec = x.sum(dim=1)
t_vec = time.perf_counter() - t0

print(f"loop: {t_loop*1e3:.1f} ms, vec: {t_vec*1e3:.3f} ms")  # vec is much faster
print(torch.allclose(loop, vec))                               # True

</details>

**Problem 3.** Type this in Colab. Demonstrate the .item() hot-loop sync trap as code. Given a tensor loss each step, accumulate a running total TWO ways: running += loss.item() (Python float) versus running += loss.detach() (stays a tensor). Use a list of 3 fake losses [torch.tensor(2.0), torch.tensor(1.0), torch.tensor(0.5)] and print the type of each running total.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Accumulate with loss.detach() and only call .item() once at the end. — _On a GPU, .item() forces a host/device sync every step; keeping the total on-device avoids the stall._
- Print type(running) for each. — _The .item() total is a Python float (synced each step); the .detach() total stays a tensor until you sync once._

**Answer:** losses = [torch.tensor(2.0), torch.tensor(1.0), torch.tensor(0.5)]

# BAD: .item() every step -> a host/device sync each iteration on GPU
running_bad = 0.0
for loss in losses:
    running_bad += loss.item()
print(type(running_bad), running_bad)      # <class 'float'> 3.5

# GOOD: stay on-device, sync ONCE at the end
running_good = torch.zeros(())
for loss in losses:
    running_good += loss.detach()
print(type(running_good), running_good.item())  # <class 'torch.Tensor'> 3.5

</details>

**Problem 4.** Type this in Colab. Implement gradient accumulation and verify the averaging detail. For model = nn.Linear(4, 1) (seed 0), micro-batch x = torch.randn(8, 4), y = torch.randn(8, 1), with accum_steps = 4: do 4 backward passes of loss / accum_steps WITHOUT stepping, then print the gradient norm. Compare it to backprop on the full 4&times; data at once.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Divide each micro-batch loss by accum_steps before backward(), accumulating grads. — _Without the division you accumulate the SUM, not the average — the gradient (and effective lr) is accum_stepsx too large._
- Compare to one backward over the concatenated 4&times; batch. — _Done right, accumulated micro-batch grads equal the gradient of one big batch._

**Answer:** torch.manual_seed(0)
model = nn.Linear(4, 1)
x = torch.randn(8, 4); y = torch.randn(8, 1)
accum_steps = 4

# Accumulate 4 identical micro-batches, scaling each loss by 1/accum_steps
model.zero_grad()
for _ in range(accum_steps):
    loss = ((model(x) - y) ** 2).mean() / accum_steps
    loss.backward()
g_accum = model.weight.grad.norm().item()

# One backward over the full 4x batch (same data repeated)
model.zero_grad()
X = x.repeat(accum_steps, 1); Y = y.repeat(accum_steps, 1)
((model(X) - Y) ** 2).mean().backward()
g_full = model.weight.grad.norm().item()

print(round(g_accum, 5), round(g_full, 5))   # equal -> averaging is correct

</details>

**Problem 5.** Type this in Colab. Build an efficient DataLoader. Wrap TensorDataset(torch.randn(256, 10), torch.randint(0, 3, (256,))) in a DataLoader with batch_size=64, num_workers=2, pin_memory=False, drop_last=True. Iterate once and print how many batches you get and the shape of the first batch's inputs. Predict the batch count.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Set num_workers>0 so worker subprocesses prefetch batches in parallel with compute. — _With num_workers=0 the GPU waits while the main process loads each batch — the classic starvation bottleneck._
- Predict the batch count: 256 // 64 = 4 with drop_last=True. — _drop_last keeps a fixed batch shape, which also avoids torch.compile recompiles._

**Answer:** from torch.utils.data import DataLoader, TensorDataset
torch.manual_seed(0)
ds = TensorDataset(torch.randn(256, 10), torch.randint(0, 3, (256,)))
loader = DataLoader(ds, batch_size=64, num_workers=2,
                    pin_memory=False, drop_last=True)
batches = list(loader)
print(len(batches))            # 4   (256 // 64, last partial dropped)
print(batches[0][0].shape)     # torch.Size([64, 10])

</details>

**Problem 6.** Type this in Colab. Move a batch to the chosen device with an asynchronous copy. Pick device = "cuda" if torch.cuda.is_available() else "cpu", make xb = torch.randn(64, 10), and copy it with xb.to(device, non_blocking=True). Print the result's .device. Explain in a comment what non_blocking=True needs to actually help.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Copy with .to(device, non_blocking=True). — _non_blocking=True lets the CPU&rarr;GPU transfer run in the background instead of stalling the host._
- Note it only overlaps when the source is in pinned (pin_memory) host memory. — _Without pinned memory the copy is synchronous anyway, so DataLoader pin_memory=True is the partner setting._

**Answer:** device = "cuda" if torch.cuda.is_available() else "cpu"
xb = torch.randn(64, 10)
xb = xb.to(device, non_blocking=True)
print(xb.device)        # cuda:0 on a GPU runtime, else cpu
# non_blocking=True only overlaps the copy when xb is in pinned host memory
# (DataLoader pin_memory=True), otherwise the transfer is synchronous.

</details>

**Problem 7.** Type this in Colab. Apply torch.compile and confirm it still produces the same numbers. Build model = nn.Sequential(nn.Linear(10, 10), nn.ReLU(), nn.Linear(10, 3)) (seed 0), get out_eager = model(x) for x = torch.randn(8, 10), then compiled = torch.compile(model) and compare compiled(x) with torch.allclose(..., atol=1e-5).

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Wrap with torch.compile(model) — one line, no other code changes. — _It traces and fuses the graph into optimized kernels; the first call compiles, later calls reuse it._
- Compare compiled vs eager output with torch.allclose. — _Compilation is a speed optimization, not a math change — the outputs must match (up to tiny numerical noise)._

**Answer:** torch.manual_seed(0)
model = nn.Sequential(nn.Linear(10, 10), nn.ReLU(), nn.Linear(10, 3))
x = torch.randn(8, 10)
out_eager = model(x)

compiled = torch.compile(model)        # PyTorch 2.x; first call compiles
out_compiled = compiled(x)
print(torch.allclose(out_eager, out_compiled, atol=1e-5))   # True

</details>

**Problem 8.** Type this in Colab. Put it together: a small profiled-style training step that avoids the sync trap. Build model = nn.Sequential(nn.Linear(10, 32), nn.ReLU(), nn.Linear(32, 3)), an Adam optimizer, and loop 50 steps over fixed xb/yb (seed 0), accumulating running with loss.detach() and calling .item() only once at the end. Print the final averaged loss.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Accumulate the loss on-device with running += loss.detach() inside the loop. — _Calling .item() every step forces a host/device sync; detaching keeps it on the GPU._
- Call .item() exactly once, after the loop. — _One sync at the end (for logging) instead of fifty in the hot loop — free speed on a real GPU._

**Answer:** torch.manual_seed(0)
model = nn.Sequential(nn.Linear(10, 32), nn.ReLU(), nn.Linear(32, 3))
opt = torch.optim.Adam(model.parameters(), lr=1e-2)
loss_fn = nn.CrossEntropyLoss()
xb = torch.randn(16, 10); yb = torch.randint(0, 3, (16,))

running = torch.zeros(())
for _ in range(50):
    opt.zero_grad()
    loss = loss_fn(model(xb), yb)
    loss.backward(); opt.step()
    running += loss.detach()           # stays on-device; NO per-step .item()
print("avg loss:", round((running / 50).item(), 4))   # one sync at the end

</details>